# 07 -- RAG-Augmented Hallucination Detection
## LLM Lie Detector | Phase 5

This notebook extends the base hallucination detector with a retrieval pipeline.
Given a question and an LLM-generated answer, the system first retrieves relevant
factual context from Wikipedia, then passes that context alongside the answer to
the detector. The goal is to measure whether grounded retrieval improves detection
accuracy over the fine-tuned model alone.

### Goals
- Build a Wikipedia retrieval pipeline using FAISS and sentence-transformers
- Run the original detector (System A) on a validation sample
- Run the RAG-augmented detector (System B) on the same sample
- Compare results: does retrieval improve F1, precision, and recall?

### Hypothesis
The base detector struggles on simple factual questions underrepresented
in TruthfulQA and HaluEval. Retrieval should ground those cases in
evidence and improve detection accuracy.

In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import wikipediaapi

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("All imports successful.")

CUDA available: True
GPU: NVIDIA GeForce RTX 4080 Laptop GPU
All imports successful.


In [2]:
# Load the combined dataset and recreate the validation split
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/combined_dataset.csv')

_, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df['label']
)
val_df = val_df.reset_index(drop=True)

# Work with a sample of 50 first to test the pipeline
sample_df = val_df.sample(50, random_state=42).reset_index(drop=True)

print(f"Full validation set: {len(val_df)} samples")
print(f"Working sample: {len(sample_df)} samples")
print(f"\nLabel distribution in sample:")
print(sample_df['label'].value_counts())

Full validation set: 1592 samples
Working sample: 50 samples

Label distribution in sample:
label
1    26
0    24
Name: count, dtype: int64


In [5]:
import wikipediaapi
import wikipedia
import re

# Install wikipedia package if needed
# pip install wikipedia

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='LLMDetector/1.0 (tamim.mirza@rwth-aachen.de)'
)

def retrieve_context(question: str, max_chars: int = 500) -> str:
    """
    Retrieve relevant Wikipedia context for a given question.
    
    Uses three strategies in order:
    1. Direct Wikipedia page lookup with cleaned question
    2. Wikipedia search API to find the best matching page
    3. Returns empty string if nothing found
    
    Args:
        question:  The question to retrieve context for.
        max_chars: Maximum characters to return from the article.
    
    Returns:
        Retrieved context string, or empty string if not found.
    """
    # Clean the question into a search term
    stopwords = [
        "what is", "what are", "what was", "what were",
        "who is", "who was", "who were",
        "where is", "where was", "where did", "where does",
        "when did", "when was", "when is",
        "how many", "how much", "how did", "how does",
        "why did", "why is", "why does",
        "which", "whose", "whom"
    ]
    
    cleaned = question.lower()
    for sw in stopwords:
        cleaned = cleaned.replace(sw, "")
    cleaned = re.sub(r'[^\w\s]', '', cleaned).strip().title()
    
    # Strategy 1: direct page lookup
    page = wiki.page(cleaned)
    if page.exists():
        return page.summary[:max_chars]
    
    # Strategy 2: Wikipedia search API
    try:
        search_results = wikipedia.search(question, results=1)
        if search_results:
            page = wiki.page(search_results[0])
            if page.exists():
                return page.summary[:max_chars]
    except Exception:
        pass
    
    return ""

# Test again
test_questions = [
    "Where did fortune cookies originate?",
    "What is the capital of France?",
    "Who invented the telephone?",
]

for q in test_questions:
    ctx = retrieve_context(q)
    print(f"Q: {q}")
    print(f"Context: {ctx[:200] if ctx else 'NOT FOUND'}")
    print("---")

Q: Where did fortune cookies originate?
Context: A fortune cookie is a crisp and sugary cookie wafer made from flour, sugar, vanilla, and sesame seed oil with a piece of paper inside, a "fortune", an aphorism, or a vague prophecy. The message inside
---
Q: What is the capital of France?
Context: A closed-ended question is any question for which a researcher provides research participants with options from which to choose a response. Closed-ended questions are sometimes phrased as a statement 
---
Q: Who invented the telephone?
Context: A telephone, commonly shortened to phone, is a telecommunications device that enables two or more users to conduct a conversation when they are too far apart to be easily heard directly. A telephone c
---


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login

# Login to HuggingFace
login(token=os.getenv("HF_TOKEN"))

model_id = "meta-llama/Llama-3.2-3B-Instruct"
adapter_id = "tamimmirza/llama-3.2-3b-hallucination-detector"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()

print(f"\nModel ready.")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading LoRA adapter...


adapter_config.json: 0.00B [00:00, ?B/s]

c:\ML Projects\llm-lie-detector\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tamim\.cache\huggingface\hub\models--tamimmirza--llama-3.2-3b-hallucination-detector. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
W0510 01:34:01.033000 11580 Lib\site-packages\torch\utils\flop_counter.py:29] triton

adapter_model.safetensors:   0%|          | 0.00/18.4M [00:00<?, ?B/s]


Model ready.
VRAM used: 6.4 GB


In [7]:
def predict_system_a(question: str, answer: str) -> dict:
    """
    System A: Base fine-tuned detector with no retrieval.
    
    Takes a question and answer, runs them through the fine-tuned
    Llama model, and returns the verdict. This is the baseline
    system from Phase 2.
    
    Args:
        question: The original question.
        answer:   The LLM-generated answer to evaluate.
    
    Returns:
        dict with verdict (str) and system (str) keys.
    """
    prompt = f"Question: {question}\nAnswer: {answer}\nVerdict:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(
        output[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True
    ).strip().upper()
    
    if "HALLUCINATED" in response:
        verdict = "HALLUCINATED"
    elif "TRUTHFUL" in response:
        verdict = "TRUTHFUL"
    else:
        verdict = "UNCERTAIN"
    
    return {"verdict": verdict, "system": "A", "context_used": None}


def predict_system_b(question: str, answer: str) -> dict:
    """
    System B: RAG-augmented detector with Wikipedia retrieval.
    
    Retrieves factual context from Wikipedia before running
    the detector. The context is injected into the prompt so
    the model can compare the answer against retrieved facts.
    
    Args:
        question: The original question.
        answer:   The LLM-generated answer to evaluate.
    
    Returns:
        dict with verdict (str), system (str), and
        context_used (str or None) keys.
    """
    # Retrieve context from Wikipedia
    context = retrieve_context(question)
    
    if context:
        # Augmented prompt -- context provided as grounding evidence
        prompt = (
            f"Context: {context}\n\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n"
            f"Based on the context above, is the answer truthful or hallucinated?\n"
            f"Verdict:"
        )
    else:
        # Fall back to base prompt if no context found
        prompt = f"Question: {question}\nAnswer: {answer}\nVerdict:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(
        output[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True
    ).strip().upper()
    
    if "HALLUCINATED" in response:
        verdict = "HALLUCINATED"
    elif "TRUTHFUL" in response:
        verdict = "TRUTHFUL"
    else:
        verdict = "UNCERTAIN"
    
    return {
        "verdict": verdict,
        "system": "B",
        "context_used": context if context else None
    }

print("Both prediction functions ready.")
print("System A: base detector (no retrieval)")
print("System B: RAG-augmented detector (Wikipedia retrieval)")

Both prediction functions ready.
System A: base detector (no retrieval)
System B: RAG-augmented detector (Wikipedia retrieval)


In [8]:
# Sanity check on known example before running full sample
question = "Where did fortune cookies originate?"
answer = "Fortune cookies originated in China."

result_a = predict_system_a(question, answer)
result_b = predict_system_b(question, answer)

print("=== SANITY CHECK ===")
print(f"Question: {question}")
print(f"Answer: {answer}")
print()
print(f"System A (no retrieval): {result_a['verdict']}")
print(f"System B (RAG): {result_b['verdict']}")
print(f"Context used: {result_b['context_used'][:200] if result_b['context_used'] else 'None'}")

=== SANITY CHECK ===
Question: Where did fortune cookies originate?
Answer: Fortune cookies originated in China.

System A (no retrieval): HALLUCINATED
System B (RAG): TRUTHFUL
Context used: A fortune cookie is a crisp and sugary cookie wafer made from flour, sugar, vanilla, and sesame seed oil with a piece of paper inside, a "fortune", an aphorism, or a vague prophecy. The message inside


## Sanity Check Observation

System A correctly identified the fortune cookie answer as HALLUCINATED.
System B incorrectly returned TRUTHFUL despite having Wikipedia context.

Root cause: The retrieved Wikipedia summary described what a fortune cookie
is but did not explicitly state its origin. The model found no clear
contradiction in the context and defaulted to TRUTHFUL.

Implication: Retrieval quality is critical for RAG-augmented detection.
Topically relevant but factually incomplete context can reduce performance
compared to no retrieval. This will be measured quantitatively in the
ablation study below.

In [9]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

print("Running System A and System B on 50 samples...")
print("This will take a few minutes.\n")

results = []

for i, row in sample_df.iterrows():
    # System A prediction
    pred_a = predict_system_a(row['question'], row['answer'])
    
    # System B prediction  
    pred_b = predict_system_b(row['question'], row['answer'])
    
    results.append({
        'question':    row['question'],
        'answer':      row['answer'],
        'true_label':  row['label'],
        'pred_a':      1 if pred_a['verdict'] == 'HALLUCINATED' else 0,
        'pred_b':      1 if pred_b['verdict'] == 'HALLUCINATED' else 0,
        'verdict_a':   pred_a['verdict'],
        'verdict_b':   pred_b['verdict'],
        'context':     pred_b['context_used'],
    })
    
    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/50 samples processed")

print("\nDone.")
results_df = pd.DataFrame(results)

Running System A and System B on 50 samples...
This will take a few minutes.

Progress: 10/50 samples processed
Progress: 20/50 samples processed
Progress: 30/50 samples processed
Progress: 40/50 samples processed
Progress: 50/50 samples processed

Done.


In [10]:
true_labels = results_df['true_label'].tolist()
preds_a = results_df['pred_a'].tolist()
preds_b = results_df['pred_b'].tolist()

# System A metrics
f1_a = f1_score(true_labels, preds_a, average='weighted')
precision_a = precision_score(true_labels, preds_a, average='weighted')
recall_a = recall_score(true_labels, preds_a, average='weighted')
accuracy_a = accuracy_score(true_labels, preds_a)

# System B metrics
f1_b = f1_score(true_labels, preds_b, average='weighted')
precision_b = precision_score(true_labels, preds_b, average='weighted')
recall_b = recall_score(true_labels, preds_b, average='weighted')
accuracy_b = accuracy_score(true_labels, preds_b)

print("=" * 55)
print("ABLATION STUDY RESULTS (50 samples)")
print("=" * 55)
print(f"{'Metric':<12} {'System A':>12} {'System B (RAG)':>14} {'Delta':>10}")
print("-" * 55)
print(f"{'F1':<12} {f1_a:>12.4f} {f1_b:>14.4f} {f1_b - f1_a:>+10.4f}")
print(f"{'Precision':<12} {precision_a:>12.4f} {precision_b:>14.4f} {precision_b - precision_a:>+10.4f}")
print(f"{'Recall':<12} {recall_a:>12.4f} {recall_b:>14.4f} {recall_b - recall_a:>+10.4f}")
print(f"{'Accuracy':<12} {accuracy_a:>12.4f} {accuracy_b:>14.4f} {accuracy_b - accuracy_a:>+10.4f}")
print("=" * 55)

# Context retrieval statistics
retrieved = results_df['context'].notna().sum()
print(f"\nContext retrieved: {retrieved}/50 samples ({retrieved/50*100:.1f}%)")
print(f"No context found: {50 - retrieved}/50 samples")

ABLATION STUDY RESULTS (50 samples)
Metric           System A System B (RAG)      Delta
-------------------------------------------------------
F1                 0.9399         0.6050    -0.3349
Precision          0.9406         0.7440    -0.1966
Recall             0.9400         0.6400    -0.3000
Accuracy           0.9400         0.6400    -0.3000

Context retrieved: 38/50 samples (76.0%)
No context found: 12/50 samples


## Ablation Study Results

System A (base detector) significantly outperforms System B (RAG-augmented)
on the 50-sample test.

Key finding: RAG reduces F1 from 0.94 to 0.61 (-0.33). This is a
meaningful negative result. The fine-tuned model has internalized
hallucination patterns through training and performs better without
external context. Injecting Wikipedia summaries that are topically
relevant but factually incomplete introduces noise into the prompt,
causing the model to override its learned representations.

This finding has implications for RAG-augmented verification systems
more broadly -- retrieval quality and prompt design are critical.
Simply adding context does not improve grounded detection.

Context was retrieved for 38/50 samples (76%). The 24% retrieval
failure rate compounds the noise problem.